<div style="width: 100%; margin-bottom: 20px;">
    <img src="../assets/banner.png" style="width: 100%; border-radius: 10px; height: 300px; object-fit: cover;">
</div>

# Phase 1 – Exploration & Validation du Dataset
## Projet ML : Prédiction des Tempêtes Géomagnétiques

**Réalisé par :** 
- AMEZIANE Oumaima
- ROHAND Douae
- MOHITO Raihana
  
**Date :** 2025-2026  
**Cours :** Machine Learning – 2ème année Cycle Ingénieurs GI  
**Encadrant :** Pr. Y. EL YOUNOUSSI

---

Le Soleil éjecte en permanence des particules chargées (vent solaire) qui, lors d'éruptions
intenses, provoquent des **tempêtes géomagnétiques** : des événements rares mais critiques
capables de perturber satellites, réseaux GPS et réseaux électriques à l'échelle mondiale.
L'objectif de ce projet est de construire un **modèle de classification binaire** capable de
prédire l'occurrence d'une tempête (indice **Kp ≥ 5**) **6 heures à l'avance**, à partir des
mesures de météo spatiale des sondes NASA/NOAA captées au point de Lagrange L1.
Ce notebook constitue la **Phase 1 — Exploration & Validation** : nous y vérifions que le
dataset (43 207 observations horaires, 2019–2023) respecte toutes les contraintes imposées
- taille, richesse des features, mix de types de variables et déséquilibre naturel de classe
(~8.4 % de tempêtes).

## 0. Imports et Configuration

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Style global des graphiques
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12

# Configuration des couleurs
COLOR_STORM = '#FF8C00'  # bordeaux (burgundy)
COLOR_CALM    = '#1a237e'   # bleu nuit
COLOR_ACCENT  = '#76ff03'   # vert aurore

TARGET = 'is_storm'

## 1. Chargement du Dataset

In [2]:
df = pd.read_csv('../data/dataset.csv')
print(f'Dataset chargé : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')

# Calcul immédiat des statistiques de classe pour les graphiques suivants
counts = df[TARGET].value_counts().sort_index()
pct_storm = (counts[1] / counts.sum() * 100) if 1 in counts else 0
pct_calm  = (counts[0] / counts.sum() * 100) if 0 in counts else 0

Dataset chargé : 43,207 lignes × 15 colonnes


## 2. Vue sur les 5 premier lignes du dataset 

In [3]:
df.head()

,timestamp,solar_wind_speed,solar_wind_density,bz_component,solar_wind_pressure,bz_min_3h,dst_index,month,sin_month,cos_month,season,hour_interval,bz_negative,is_solar_maximum,is_storm
0,2019-01-01 08:00:00,459.0,4.0,1.9,842724.0,-1.0,-9.0,1,0.5,0.866025,hiver,6h-9h,0.0,0,0
1,2019-01-01 09:00:00,453.0,4.5,2.5,923440.5,-1.0,-7.0,1,0.5,0.866025,hiver,9h-12h,0.0,0,0
2,2019-01-01 10:00:00,446.0,5.2,-0.1,1034363.2,-0.1,-4.0,1,0.5,0.866025,hiver,9h-12h,1.0,0,0
3,2019-01-01 11:00:00,438.0,4.6,-3.4,882482.4,-3.4,-9.0,1,0.5,0.866025,hiver,9h-12h,1.0,0,0
4,2019-01-01 12:00:00,433.0,4.3,-3.4,806202.7,-3.4,-12.0,1,0.5,0.866025,hiver,12h-15h,1.0,0,0


**Interprétation :** Les 5 premières lignes confirment la structure attendue du dataset.
Les données sont horaires, débutant le 1er janvier 2019. On observe que les features
physiques (solar_wind_speed ≈ 430-460 km/s, bz_component légèrement négatif, dst_index
entre -12 et -4) correspondent à une période de vent solaire calme — ce qui est cohérent
avec is_storm = 0 sur ces lignes. La colonne `timestamp` sera exclue de la modélisation
(identifiant temporel, pas une feature prédictive).

## 3. Vue d'ensemble : `df.info()`

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 43207 entries, 0 to 43206
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   timestamp            43207 non-null  str    
 1   solar_wind_speed     43207 non-null  float64
 2   solar_wind_density   43207 non-null  float64
 3   bz_component         43207 non-null  float64
 4   solar_wind_pressure  43207 non-null  float64
 5   bz_min_3h            43207 non-null  float64
 6   dst_index            43207 non-null  float64
 7   month                43207 non-null  int64  
 8   sin_month            43207 non-null  float64
 9   cos_month            43207 non-null  float64
 10  season               43207 non-null  str    
 11  hour_interval        43207 non-null  str    
 12  bz_negative          43207 non-null  float64
 13  is_solar_maximum     43207 non-null  int64  
 14  is_storm             43207 non-null  int64  
dtypes: float64(9), int64(3), str(3)
memory usage: 6

**Interprétation :** Le dataset contient **43 207 entrées** sans aucune valeur manquante
(43 207 non-null partout) — aucun traitement d'imputation ne sera nécessaire.
On distingue **3 types de données** :
- `float64` (9 colonnes) : features physiques continues — solar_wind_speed, bz_component, etc.
- `int64` (3 colonnes) : month, is_solar_maximum, is_storm (la cible).
- `str` (3 colonnes) : timestamp, season, hour_interval — variables catégorielles textuelles.

La mémoire occupée est légère (6.2 MB), le dataset est parfaitement exploitable.

## 4. Statistiques descriptives : `df.describe()`

In [5]:
df.describe()

,solar_wind_speed,solar_wind_density,bz_component,solar_wind_pressure,bz_min_3h,dst_index,month,sin_month,cos_month,bz_negative,is_solar_maximum,is_storm
count,43207.000000,43207.000000,43207.000000,4.320700e+04,43207.000000,43207.000000,43207.000000,4.320700e+04,4.320700e+04,43207.000000,43207.000000,43207.000000
mean,410.103502,6.190365,-0.130196,9.489742e+05,-1.247830,-8.622052,6.525517,-4.955405e-03,-1.558478e-03,0.525771,0.399519,0.084107
std,86.251157,4.733501,2.756907,6.644353e+05,2.613833,14.622127,3.447336,7.068398e-01,7.073710e-01,0.499341,0.489805,0.277551
min,259.000000,0.100000,-32.400000,1.857610e+04,-32.400000,-213.000000,1.000000,-1.000000e+00,-1.000000e+00,0.000000,0.000000,0.000000
25%,344.000000,3.200000,-1.600000,5.628777e+05,-2.500000,-15.000000,4.000000,-8.660254e-01,-8.660254e-01,0.000000,0.000000,0.000000
50%,391.000000,4.900000,-0.200000,7.879896e+05,-1.200000,-6.000000,7.000000,-2.449294e-16,-1.836970e-16,1.000000,0.000000,0.000000
75%,461.000000,7.700000,1.200000,1.126735e+06,0.100000,0.000000,10.000000,5.000000e-01,8.660254e-01,1.000000,1.000000,0.000000
max,846.000000,84.400000,31.300000,1.211050e+07,27.500000,42.000000,12.000000,1.000000e+00,1.000000e+00,1.000000,1.000000,1.000000


**Interprétation des statistiques descriptives :**

- **solar_wind_speed** : moyenne 410 km/s, min 259, max 846. La forte dispersion (std=86)
  indique des épisodes de vent solaire rapide — caractéristique des tempêtes géomagnétiques.
- **bz_component** : moyenne quasi-nulle (-0.13 nT), mais min à -32.4 nT. Les valeurs
  très négatives de Bz sont le principal déclencheur de tempêtes (couplage magnétique
  avec la Terre) — feature potentiellement très discriminante.
- **dst_index** : moyenne -8.6 nT, min -213 nT. Les valeurs extrêmes négatives signalent
  des tempêtes intenses (seuil storm : dst < -50 nT). La forte asymétrie confirme le
  déséquilibre naturel de la cible.
- **solar_wind_pressure** : très grande plage (min ≈ 18 571, max ≈ 12M nPa), std élevée —
  présence probable d'outliers liés aux éjections de masse coronale (CME).
- **month** : moyenne 6.5, distribution quasi-uniforme sur 1–12 → couverture temporelle
  équilibrée sur toute l'année.
- **bz_negative** : moyenne 0.53 → environ 53% des observations ont un Bz négatif,
  ce qui est cohérent avec une légère prépondérance des périodes calmes.

## 5. Validation des Contraintes du Projet

### 5.1 – Contrainte 1 : Taille ≥ 10 000 lignes

In [6]:
n_rows = df.shape[0]
seuil_lignes = 10_000

statut = 'VALIDÉE' if n_rows >= seuil_lignes else ' NON VALIDÉE'
print(f'Contrainte 1 — Nombre de lignes : {n_rows:,} (seuil : {seuil_lignes:,}) → {statut}')

Contrainte 1 — Nombre de lignes : 43,207 (seuil : 10,000) → VALIDÉE


### 5.2 – Contrainte 2 : Nombre de features ≥ 8

In [7]:
# On exclut la colonne cible du comptage des features
TARGET = 'is_storm'
features = [c for c in df.columns if c != TARGET]
n_features = len(features)
seuil_features = 8

statut = ' VALIDÉE' if n_features >= seuil_features else ' NON VALIDÉE'
print(f'Contrainte 2 — Nombre de features : {n_features} (seuil : {seuil_features}) → {statut}')
print(f'\nListe des features : {features}')

Contrainte 2 — Nombre de features : 14 (seuil : 8) →  VALIDÉE

Liste des features : ['timestamp', 'solar_wind_speed', 'solar_wind_density', 'bz_component', 'solar_wind_pressure', 'bz_min_3h', 'dst_index', 'month', 'sin_month', 'cos_month', 'season', 'hour_interval', 'bz_negative', 'is_solar_maximum']


### 5.3 – Contrainte 3 : Mix de variables numériques et catégorielles

In [8]:
numeriques   = df[features].select_dtypes(include=['number']).columns.tolist()
categorielles = df[features].select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

print(f'Variables numériques  ({len(numeriques)}) : {numeriques}')
print(f'Variables catégorielles ({len(categorielles)}) : {categorielles}')

statut = ' VALIDÉE' if (len(numeriques) > 0 and len(categorielles) > 0) else ' NON VALIDÉE'
print(f'\nContrainte 3 — Mix num./cat. → {statut}')

Variables numériques  (11) : ['solar_wind_speed', 'solar_wind_density', 'bz_component', 'solar_wind_pressure', 'bz_min_3h', 'dst_index', 'month', 'sin_month', 'cos_month', 'bz_negative', 'is_solar_maximum']
Variables catégorielles (3) : ['timestamp', 'season', 'hour_interval']

Contrainte 3 — Mix num./cat. →  VALIDÉE


## 6. Visualisations du Déséquilibre de Classe

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Distribution de la Variable Cible : is_storm', fontsize=15, fontweight='bold', y=1.02)

labels = ['Calme (0)', 'Tempête (1)']
values = [counts[0], counts[1]]
colors = [COLOR_CALM, COLOR_STORM]

# --- Bar Plot ---
bars = axes[0].bar(labels, values, color=colors, edgecolor='white', linewidth=1.5, width=0.5)
axes[0].set_title('Bar Plot – Effectifs par classe', fontweight='bold')
axes[0].set_ylabel('Nombre d\'échantillons')
axes[0].set_xlabel('Classe')
for bar, val, pct in zip(bars, values, [pct_calm, pct_storm]):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 200,
        f'{val:,}\n({pct:.1f}%)',
        ha='center', va='bottom', fontweight='bold', fontsize=11
    )
axes[0].set_ylim(0, max(values) * 1.15)

# Zone de déséquilibre accepté
axes[0].axhline(y=counts.sum() * 0.05, color=COLOR_ACCENT, linestyle='--', alpha=0.7, label='Seuil 5%')
axes[0].axhline(y=counts.sum() * 0.25, color='orange', linestyle='--', alpha=0.7, label='Seuil 25%')
axes[0].legend(fontsize=9)

# --- Pie Chart ---
wedges, texts, autotexts = axes[1].pie(
    values,
    labels=labels,
    colors=colors,
    autopct='%1.2f%%',
    startangle=90,
    explode=(0, 0.07),
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    textprops={'fontsize': 12}
)
for at in autotexts:
    at.set_fontweight('bold')
    at.set_fontsize(13)
axes[1].set_title('Pie Chart – Proportion par classe', fontweight='bold')

plt.tight_layout()
plt.savefig('../data/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure sauvegardée → data/class_distribution.png')

**Interprétation du déséquilibre de classe :**

Le dataset présente un déséquilibre naturel et contrôlé :
- **Classe 0 – Calme** : 39 573 échantillons (**91.6 %**)
- **Classe 1 – Tempête** : 3 634 échantillons (**8.4 %**)
- **Ratio de déséquilibre** : environ 1:11

 Ce ratio de **8.4 %** respecte parfaitement la contrainte imposée (entre 5 % et 25 %).
Ce déséquilibre est **naturel et scientifiquement justifié** : les tempêtes géomagnétiques
sont des événements rares par nature. Il n'a pas été artificiellement forcé.

**Conséquence pour la Phase 2 :** L'accuracy seule sera une métrique trompeuse
(un modèle naïf prédisant toujours 0 obtiendrait 91.6% d'accuracy). On privilégiera
le **Recall** et le **F1-score** comme métriques principales, avec des techniques
de rééchantillonnage (SMOTE) ou de pondération des classes (`class_weight='balanced'`).

## 7. Exploration des Features

### 7.1 – Distribution des features numériques

In [ ]:
if numeriques:
    n_cols = 3
    n_rows_plot = (len(numeriques) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows_plot, n_cols, figsize=(15, 4 * n_rows_plot))
    axes = axes.flatten() if n_rows_plot > 1 else [axes]

    for i, col in enumerate(numeriques):
        for val, color, label in zip([0, 1], [COLOR_CALM, COLOR_STORM], ['Calme', 'Tempête']):
            axes[i].hist(
                df[df[TARGET] == val][col].dropna(),
                bins=40, alpha=0.6, color=color, label=label, density=True
            )
        axes[i].set_title(col, fontweight='bold')
        axes[i].legend(fontsize=9)
        axes[i].set_xlabel(col)
        axes[i].set_ylabel('Densité')

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('Distribution des features numériques par classe', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

**Interprétation des distributions par classe :**

- **solar_wind_speed** : la classe Tempête (orange) est décalée vers des vitesses plus
  élevées (500-700 km/s) par rapport à la classe Calme. Le vent solaire rapide est un
  précurseur clé des aurores — feature discriminante.
- **solar_wind_density** : distributions très similaires entre classes, légère surreprésentation
  des densités élevées en période de tempête.
- **bz_component** : la classe Tempête présente plus de valeurs négatives prononcées.
  Bz négatif = condition nécessaire au déclenchement d'une tempête géomagnétique.
- **solar_wind_pressure** : pic très concentré pour les deux classes, quelques valeurs
  extrêmes côté Tempête (CME).
- **bz_min_3h** : la classe Tempête s'étale davantage vers les valeurs négatives — confirme
  l'importance du minimum de Bz sur une fenêtre glissante de 3h.
- **dst_index** : distribution clairement **bimodale** — la classe Tempête est très décalée
  vers les valeurs négatives (-50 à -213 nT). C'est la feature la plus corrélée à la cible.
- **bz_negative / is_solar_maximum** : features binaires (0/1). Les tempêtes surviennent
  préférentiellement quand bz_negative=1 et pendant le maximum solaire (is_solar_maximum=1).

### 7.2 – Distribution des features catégorielles

In [ ]:
if categorielles:
    # Filtre de sécurité pour éviter de bloquer sur 'timestamp'
    cat_to_plot = [col for col in categorielles if df[col].nunique() <= 20]
    
    n_cols = 2
    n_rows_plot = (len(cat_to_plot) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows_plot, n_cols, figsize=(14, 4 * n_rows_plot))
    axes = np.array(axes).flatten()

    for i, col in enumerate(cat_to_plot):
        ct = pd.crosstab(df[col], df[TARGET])
        ct.columns = ['Calme', 'Tempête']
        ct.plot(kind='bar', ax=axes[i], color=[COLOR_CALM, COLOR_STORM],
                edgecolor='white', width=0.7)
        axes[i].set_title(col, fontweight='bold')
        axes[i].set_xlabel('')
        axes[i].set_ylabel('Effectifs')
        axes[i].tick_params(axis='x', rotation=30)
        axes[i].legend(fontsize=9)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('Distribution des features catégorielles par classe', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print('Aucune feature catégorielle détectée.')

**Interprétation des features catégorielles :**

- **season** : les tempêtes sont plus fréquentes en **automne** et **printemps**.
  C'est un phénomène connu en géophysique (équinoxes de Russell-McPherron) : l'alignement
  géométrique Terre-Soleil favorise le couplage magnétique aux équinoxes.
  → `season` est une feature pertinente, confirmée par la physique solaire.

- **hour_interval** : la distribution des tempêtes est relativement **uniforme sur 24h**,
  avec une légère tendance pour les plages 0h-9h. Les tempêtes géomagnétiques ne dépendent
  pas fortement de l'heure locale — ce que confirme ce graphique.
  → `hour_interval` apporte une information complémentaire mais moins discriminante que
  les features physiques.

### 7.3 – Matrice de corrélation (features numériques)

In [ ]:
if len(numeriques) >= 2:
    corr_matrix = df[numeriques + [TARGET]].corr()

    fig, ax = plt.subplots(figsize=(10, 8))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(
        corr_matrix, mask=mask, annot=True, fmt='.2f',
        cmap='coolwarm', center=0, linewidths=0.5,
        ax=ax, square=True
    )
    ax.set_title('Matrice de corrélation (features numériques + cible)', fontweight='bold', fontsize=13)
    plt.tight_layout()
    plt.show()

**Interprétation de la matrice de corrélation :**

Corrélations avec la cible `is_storm` :
- **dst_index (-0.34)** : corrélation négative la plus forte → plus le dst est bas,
  plus la probabilité de tempête est élevée. Feature la plus prédictive.
- **solar_wind_speed (+0.29)** : vent solaire rapide associé aux tempêtes.
- **is_solar_maximum (+0.21)** : les tempêtes sont plus probables en phase de maximum
  du cycle solaire de 11 ans.
- **solar_wind_pressure (+0.18)** : pression élevée liée aux éjections de masse coronale.
- **bz_component et bz_min_3h** : faibles corrélations linéaires avec is_storm (-0.02, -0.07),
  mais ces features sont probablement non-linéairement importantes → les modèles
  ensemblistes (Random Forest, XGBoost) pourront mieux les exploiter.

Corrélations inter-features notables :
- **solar_wind_density / solar_wind_pressure (+0.81)** : forte colinéarité — à surveiller
  (risque de redondance d'information).
- **bz_component / bz_min_3h (+0.77)** : cohérent, le minimum sur 3h dépend du Bz courant.
- **bz_negative / bz_component (-0.70)** : logique, bz_negative est un indicateur binaire
  dérivé de bz_component.
- **sin_month / cos_month (-0.76)** : déphasage mathématique attendu (encodage cyclique).

**Conclusion :** Pas de feature inutile détectée, mais deux paires de features très
corrélées à surveiller en Phase 2 (possible suppression ou PCA).